In [1]:
import pandas as pd
import duckdb
import re
from itertools import combinations
from optbinning import OptimalBinning
from joblib import Parallel, delayed

class StrategicSegmentBuilder:
    def __init__(self, target, n_jobs=6, min_sample_size=1000, min_lift=2.0):
        self.target = target
        self.n_jobs = n_jobs
        self.min_sample_size = min_sample_size
        self.min_lift = min_lift
        self.segments = []
        
    def compute_iv_ranking(self, df):
        def _get_iv(col):
            try:
                dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
                optb = OptimalBinning(name=col, dtype=dtype)
                optb.fit(df[col].values, df[self.target].values)
                iv = optb.binning_table.build().IV.iloc[-1]
                return {"variable": col, "iv": iv * 100}
            except Exception:
                return {"variable": col, "iv": 0}

        cols = [c for c in df.columns if c != self.target]
        results = Parallel(n_jobs=self.n_jobs)(delayed(_get_iv)(col) for col in cols)
        return pd.DataFrame(results).sort_values("iv", ascending=False).reset_index(drop=True)

    def create_binned_df(self, df, variables):
        binned_df = pd.DataFrame(index=df.index)
        for col in variables:
            dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
            optb = OptimalBinning(name=col, dtype=dtype)
            optb.fit(df[col].values, df[self.target].values)
            
            # Wrap in Categorical for memory efficiency on large DataFrames
            transformed_bins = optb.transform(df[col], metric="bins")
            binned_df[col] = pd.Categorical(transformed_bins)
            
        binned_df[self.target] = df[self.target].values
        return binned_df

    def _agg_combinations(self, binned_df, combo_list, base_rate):
        def _process_combo(combo):
            summary = binned_df.groupby(list(combo), observed=True).agg(
                count=(self.target, "size"), 
                events=(self.target, "sum")
            ).reset_index()
            
            summary = summary[summary["count"] >= self.min_sample_size].copy()
            if summary.empty: return None
            
            summary["rate"] = (summary["events"] / summary["count"]) * 100
            summary["lift"] = summary["rate"] / (base_rate * 100)
            
            summary = summary[summary["lift"] >= self.min_lift]
            if summary.empty: return None
            
            rule_parts = [f"{c}=" + summary[c].astype(str) for c in combo]
            summary["rule"] = rule_parts[0]
            for rp in rule_parts[1:]:
                summary["rule"] += " & " + rp
                
            # Attach the combo variable tuple so the Apriori logic can track survivors
            summary["combo_vars"] = [combo] * len(summary)
                
            return summary[["rule", "count", "rate", "lift", "combo_vars"]]

        results = Parallel(n_jobs=self.n_jobs)(delayed(_process_combo)(c) for c in combo_list)
        valid_results = [r for r in results if r is not None]
        if not valid_results: return pd.DataFrame()
        return pd.concat(valid_results, ignore_index=True)

    def parse_rule_to_sql(self, rule_str):
        parts = [p.strip() for p in rule_str.split("&")]
        sql_conditions = []
        
        for part in parts:
            col, interval = part.split("=", 1)
            col, interval = col.strip(), interval.strip()
            
            if interval.startswith("[") and "'" in interval:
                items = re.findall(r"'(.*?)'", interval)
                formatted_items = ", ".join([f"'{i}'" for i in items])
                sql_conditions.append(f"{col} IN ({formatted_items})")
                continue
                
            if interval in ["Special", "Missing"]:
                sql_conditions.append(f"{col} IS NULL")
                continue
                
            if interval.startswith("[") or interval.startswith("("):
                left_bracket, right_bracket = interval[0], interval[-1]
                lower_str, upper_str = [x.strip() for x in interval[1:-1].split(",", 1)]
                
                col_conds = []
                if lower_str.lower() != '-inf':
                    op = '>=' if left_bracket == '[' else '>'
                    col_conds.append(f"{col} {op} {lower_str}")
                if upper_str.lower() != 'inf':
                    op = '<=' if right_bracket == ']' else '<'
                    col_conds.append(f"{col} {op} {upper_str}")
                    
                if col_conds:
                    sql_conditions.append(" AND ".join(col_conds))
                    
        return " AND ".join(f"({cond})" if "AND" in cond else cond for cond in sql_conditions)

    def extract_segments(self, df, top_n_vars=15, max_segments=10):
        current_df = df.copy()
        
        for i in range(1, max_segments + 1):
            base_rate = current_df[self.target].mean()
            if base_rate == 0 or len(current_df) < self.min_sample_size:
                break
                
            print(f"\n--- Iteration {i} | Remaining Rows: {len(current_df)} | Base Rate: {base_rate*100:.2f}% ---")
            
            iv_ranking = self.compute_iv_ranking(current_df)
            top_vars = iv_ranking.head(top_n_vars)["variable"].tolist()
            binned_df = self.create_binned_df(current_df, top_vars)
            
            all_rules = []
            
            # --- APRIORI LEVEL 1 ---
            combos_1 = [(c,) for c in top_vars]
            res_1 = self._agg_combinations(binned_df, combos_1, base_rate)
            if res_1.empty:
                print("No rules found meeting criteria. Stopping.")
                break
                
            all_rules.append(res_1)
            valid_1way_vars = {c[0] for c in res_1["combo_vars"]}
            
            # --- APRIORI LEVEL 2 ---
            valid_2way_sets = set()
            if len(valid_1way_vars) >= 2:
                combos_2 = list(combinations(valid_1way_vars, 2))
                res_2 = self._agg_combinations(binned_df, combos_2, base_rate)
                
                if not res_2.empty:
                    all_rules.append(res_2)
                    valid_2way_sets = {frozenset(c) for c in res_2["combo_vars"]}
            
            # --- APRIORI LEVEL 3 ---
            if valid_2way_sets and len(valid_1way_vars) >= 3:
                combos_3 = []
                for combo in combinations(valid_1way_vars, 3):
                    # Apriori Property: A 3-way combo is only valid if ALL of its 2-way subsets are valid
                    pairs = [frozenset(p) for p in combinations(combo, 2)]
                    if all(p in valid_2way_sets for p in pairs):
                        combos_3.append(combo)
                        
                if combos_3:
                    res_3 = self._agg_combinations(binned_df, combos_3, base_rate)
                    if not res_3.empty:
                        all_rules.append(res_3)
            
            # Combine all survived tiers and find the best
            shortlisted = pd.concat(all_rules, ignore_index=True)
            shortlisted = shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
            
            best_rule = shortlisted.loc[0, "rule"]
            best_sql = self.parse_rule_to_sql(best_rule)
            
            self.segments.append({
                "segment_id": i,
                "rule_string": best_rule,
                "sql_filter": best_sql,
                "count": shortlisted.loc[0, "count"],
                "rate": shortlisted.loc[0, "rate"],
                "lift": shortlisted.loc[0, "lift"]
            })
            
            print(f"Segment {i} Found: {best_sql}")
            
            # Filter Out Captured Population
            current_df = duckdb.query(f"SELECT * FROM current_df WHERE NOT ({best_sql})").df()
            
        return pd.DataFrame(self.segments).drop(columns=["combo_vars"], errors="ignore")

    def evaluate_final_coverage(self, original_df):
        if not self.segments:
            return "No segments to evaluate."
            
        case_statements = []
        for seg in self.segments:
            case_statements.append(f"WHEN {seg['sql_filter']} THEN {seg['segment_id']}")
            
        case_sql = "\n".join(case_statements)
        
        final_query = f"""
        SELECT 
            CASE 
                {case_sql}
                ELSE 0 
            END AS segment, 
            COUNT(*) AS total_count,
            SUM("{self.target}") AS target_events,
            (SUM("{self.target}") * 100.0 / COUNT(*)) AS response_rate
        FROM original_df
        GROUP BY 1
        ORDER BY segment
        """
        
        return duckdb.query(final_query).df()

# --- Execution ---
builder = StrategicSegmentBuilder(target="Class", n_jobs=6)
data = pd.read_csv("creditcard.csv", engine='pyarrow')
# data = data.drop(columns=["ID"])

segments_df = builder.extract_segments(data, top_n_vars=15)
final_eval = builder.evaluate_final_coverage(data)
print(final_eval)


--- Iteration 1 | Remaining Rows: 284807 | Base Rate: 0.17% ---
Segment 1 Found: V16 < -1.49 AND V11 >= 1.61

--- Iteration 2 | Remaining Rows: 283713 | Base Rate: 0.05% ---
Segment 2 Found: V14 < -1.42 AND V4 >= 1.97

--- Iteration 3 | Remaining Rows: 282640 | Base Rate: 0.03% ---
Segment 3 Found: V14 < -1.37 AND (V20 >= -0.04 AND V20 < 0.59) AND V3 < -1.98

--- Iteration 4 | Remaining Rows: 281573 | Base Rate: 0.02% ---
Segment 4 Found: V7 >= 1.29 AND V4 >= 2.47

--- Iteration 5 | Remaining Rows: 280544 | Base Rate: 0.02% ---
Segment 5 Found: V21 < -0.38 AND V4 >= 2.37

--- Iteration 6 | Remaining Rows: 279440 | Base Rate: 0.02% ---
Segment 6 Found: V13 < -1.52 AND (V4 >= 0.76 AND V4 < 1.66) AND (V25 >= -0.82 AND V25 < -0.07)

--- Iteration 7 | Remaining Rows: 278448 | Base Rate: 0.02% ---
Segment 7 Found: V14 < -1.34 AND (V4 >= 1.16 AND V4 < 1.66)

--- Iteration 8 | Remaining Rows: 277306 | Base Rate: 0.01% ---
Segment 8 Found: (V26 >= -0.16 AND V26 < -0.11) AND V4 >= 1.66

--- Ite

In [2]:
data.Class.value_counts(normalize = True)*100

Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64

In [3]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in segments_df.iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   V16=(-inf, -1.49) & V11=[1.61, inf)
SQL Filter: V16 < -1.49 AND V11 >= 1.61
--------------------------------------------------
Segment ID: 2
Raw Rule:   V14=(-inf, -1.42) & V4=[1.97, inf)
SQL Filter: V14 < -1.42 AND V4 >= 1.97
--------------------------------------------------
Segment ID: 3
Raw Rule:   V14=(-inf, -1.37) & V20=[-0.04, 0.59) & V3=(-inf, -1.98)
SQL Filter: V14 < -1.37 AND (V20 >= -0.04 AND V20 < 0.59) AND V3 < -1.98
--------------------------------------------------
Segment ID: 4
Raw Rule:   V7=[1.29, inf) & V4=[2.47, inf)
SQL Filter: V7 >= 1.29 AND V4 >= 2.47
--------------------------------------------------
Segment ID: 5
Raw Rule:   V21=(-inf, -0.38) & V4=[2.37, inf)
SQL Filter: V21 < -0.38 AND V4 >= 2.37
--------------------------------------------------
Segment ID: 6
Raw Rule:   V13=(-inf, -1.52) & V4=[0.76, 1.66) & V25=[-0.82, -0.07)
SQL Filter: V13 < -1.52 AND (V4 >= 0.76 AND V4 < 1.66) AND (V25 >= -0.82 AND V25

In [4]:
final_eval

,segment,total_count,target_events,response_rate
0,0,274236,29.0,0.010575
1,1,1094,338.0,30.895795
2,2,1073,83.0,7.735322
3,3,1067,9.0,0.843486
4,4,1029,7.0,0.680272
5,5,1104,5.0,0.452899
6,6,992,5.0,0.504032
7,7,1142,4.0,0.350263
8,8,882,4.0,0.453515
9,9,1117,5.0,0.447628
